# 2/2 — Train and evaluate ViSoBERT BASE

Attach the saved output of notebook 1 as a Kaggle Input. Select **GPU T4 x2**, enable Internet, add the secret `HF_TOKEN`, then use **Run All**. The notebook automatically finds `vihsd_manifest.json`.

In [ ]:
# ===== USER SETTINGS =====
SEED = 42
EPOCHS = 10.0          # use 0.02 for the first smoke test
EVALUATE_TEST = True   # set False while changing hyperparameters
MICRO_BATCH = 8        # if OOM: 4
GRAD_ACCUMULATION = 5  # if MICRO_BATCH=4: use 10
WEIGHT_DECAY = 0.0     # closer to SimpleTransformers classification default

# Must run before importing torch or transformers.
import os, sys, subprocess
from datetime import datetime, timezone
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert MICRO_BATCH * GRAD_ACCUMULATION == 40
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'datasets', 'accelerate', 'sentencepiece',
    'protobuf', 'huggingface_hub', 'scikit-learn', 'pandas', 'pyarrow',
], check=True)
RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = Path('/kaggle/working') / f'visobert_base_seed{SEED}_{RUN_TAG}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
print({'run_dir': str(RUN_DIR), 'effective_batch': MICRO_BATCH * GRAD_ACCUMULATION})

In [ ]:
import hashlib, json
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
from transformers import AutoTokenizer, set_seed

assert torch.cuda.is_available(), 'Enable GPU T4 x2 in Kaggle Settings.'
assert torch.cuda.device_count() == 1, 'Restart and Run All from the first cell.'
set_seed(SEED)
try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as exc:
    raise RuntimeError(
        'Create a Kaggle Secret named HF_TOKEN and enable it for this notebook.'
    ) from exc
if not HF_TOKEN or not HF_TOKEN.startswith('hf_'):
    raise RuntimeError('HF_TOKEN is missing or has an invalid format.')
try:
    HF_ACCOUNT = HfApi(token=HF_TOKEN).whoami()['name']
except Exception as exc:
    raise RuntimeError('HF_TOKEN authentication failed; replace the Kaggle secret.') from exc
manifest_paths = list(Path('/kaggle/input').rglob('vihsd_manifest.json'))
if len(manifest_paths) != 1:
    raise RuntimeError(f'Attach exactly one prepared ViHSD output; found {manifest_paths}')
manifest_path = manifest_paths[0]
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['artifact'] == 'vihsd_prepared_v1'
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
frames = {}
for split, spec in manifest['splits'].items():
    path = manifest_path.parent / spec['file']
    assert sha256(path) == spec['sha256']
    frame = pd.read_parquet(path)
    assert len(frame) == spec['rows']
    assert set(frame['labels'].unique()) <= {0, 1, 2}
    frames[split] = frame
MODEL_ID, MODEL_SHA = manifest['model_id'], manifest['model_revision']
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_SHA, token=HF_TOKEN
)
def make_dataset(frame):
    dataset = Dataset.from_dict({
        'text': frame['text'].astype(str).tolist(),
        'labels': frame['labels'].astype(int).tolist(),
    })
    return dataset.map(
        lambda batch: tokenizer(
            batch['text'], truncation=True, max_length=128, padding='max_length'
        ), batched=True, remove_columns=['text'],
    )
train_ds = make_dataset(frames['train'])
validation_ds = make_dataset(frames['validation'])
print({
    'gpu': torch.cuda.get_device_name(0), 'model_revision': MODEL_SHA,
    'dataset_revision': manifest['dataset_revision'],
    'rows': {split: len(frame) for split, frame in frames.items()},
})

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification, Trainer, TrainingArguments,
    default_data_collator,
)
LABEL2ID = {'CLEAN': 0, 'OFFENSIVE': 1, 'HATE': 2}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, revision=MODEL_SHA, token=HF_TOKEN, num_labels=3,
    label2id=LABEL2ID, id2label=ID2LABEL, attn_implementation='eager',
)
def compute_metrics(prediction):
    predicted = np.asarray(prediction.predictions).argmax(axis=-1)
    labels = np.asarray(prediction.label_ids)
    _, _, f1, support = precision_recall_fscore_support(
        labels, predicted, labels=[0, 1, 2], zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(labels, predicted)),
        'macro_f1': float(f1.mean()),
        'weighted_f1': float(np.average(f1, weights=support)),
    }
args = TrainingArguments(
    output_dir=str(RUN_DIR / 'checkpoints'),
    num_train_epochs=EPOCHS, learning_rate=2e-5,
    per_device_train_batch_size=MICRO_BATCH, per_device_eval_batch_size=16,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    optim='adamw_torch', adam_epsilon=1e-8, weight_decay=WEIGHT_DECAY,
    lr_scheduler_type='linear', warmup_ratio=0.06, max_grad_norm=1.0,
    eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='macro_f1',
    greater_is_better=True, save_total_limit=2, fp16=True,
    seed=SEED, data_seed=SEED, report_to='none', logging_steps=50,
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=validation_ds,
    processing_class=tokenizer, data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()
trainer.save_model(RUN_DIR / 'best_model')
tokenizer.save_pretrained(RUN_DIR / 'best_model')
validation_output = trainer.predict(validation_ds)
print({'best_checkpoint': trainer.state.best_model_checkpoint, **compute_metrics(validation_output)})

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

evaluation_split = 'test' if EVALUATE_TEST else 'validation'
evaluation_output = trainer.predict(make_dataset(frames['test'])) if EVALUATE_TEST else validation_output
truth = np.asarray(evaluation_output.label_ids)
logits = np.asarray(evaluation_output.predictions)
predicted = logits.argmax(axis=-1)
metrics = compute_metrics(evaluation_output)
report = classification_report(
    truth, predicted, labels=[0, 1, 2],
    target_names=['CLEAN', 'OFFENSIVE', 'HATE'], zero_division=0, output_dict=True,
)
matrix = confusion_matrix(truth, predicted, labels=[0, 1, 2])
paper = {'accuracy': 0.8851, 'weighted_f1': 0.8831, 'macro_f1': 0.6877}
comparison = {key: {'ours': float(metrics[key]), 'paper': value,
                    'delta': float(metrics[key] - value)}
              for key, value in paper.items()}
print(json.dumps({'split': evaluation_split, 'comparison': comparison}, indent=2))
display(pd.DataFrame(report).T)
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(matrix, cmap='Blues')
for row in range(3):
    for column in range(3):
        ax.text(column, row, int(matrix[row, column]), ha='center', va='center')
ax.set_xticks(range(3), ['CLEAN', 'OFFENSIVE', 'HATE'])
ax.set_yticks(range(3), ['CLEAN', 'OFFENSIVE', 'HATE'])
ax.set(xlabel='Predicted', ylabel='True', title=f'ViSoBERT BASE — {evaluation_split}')
fig.colorbar(image, ax=ax); plt.tight_layout(); plt.show()
probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()
pd.DataFrame({
    'y_true': truth, 'y_pred': predicted,
    'p_clean': probabilities[:, 0], 'p_offensive': probabilities[:, 1],
    'p_hate': probabilities[:, 2],
}).to_parquet(RUN_DIR / f'predictions_{evaluation_split}.parquet', index=False)
result = {
    'model_id': MODEL_ID, 'model_revision': MODEL_SHA,
    'dataset_revision': manifest['dataset_revision'],
    'seed': SEED, 'epochs': EPOCHS,
    'effective_batch': MICRO_BATCH * GRAD_ACCUMULATION,
    'evaluation_split': evaluation_split, 'metrics': metrics,
    'paper_comparison': comparison, 'run_dir': str(RUN_DIR),
}
(RUN_DIR / 'result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
print(f'SUCCESS: {RUN_DIR}')
print('Use Save Version to preserve the model and results.')